In [1]:
import os
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CreditCardFraudDetection-SQL") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created Successfully!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/29 00:12:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session Created Successfully!


26/03/29 00:12:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
from pyspark.sql.functions import lit

# Load original full files (100% data)
df_train = spark.read.csv("../data/fraudTrain.csv", header=True, inferSchema=True)
df_test  = spark.read.csv("../data/fraudTest.csv", header=True, inferSchema=True)

# Union = 100% full dataset
df_full = df_train.union(df_test)

# Create Temp View for SQL queries
df_full.createOrReplaceTempView("fraud_transactions")

print("Full dataset loaded!")
print("Total rows:", df_full.count())
print("Temp view 'fraud_transactions' created!")

Full dataset loaded!
Total rows: 1852394
Temp view 'fraud_transactions' created!


In [3]:
print("Query 1: Basic Fraud Overview ")
spark.sql("""
    SELECT 
        is_fraud,
        COUNT(*) as total_transactions,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
    FROM fraud_transactions
    GROUP BY is_fraud
    ORDER BY is_fraud
""").show()

Query 1: Basic Fraud Overview 


+--------+------------------+----------+
|is_fraud|total_transactions|percentage|
+--------+------------------+----------+
|       0|           1842743|     99.48|
|       1|              9651|      0.52|
+--------+------------------+----------+



In [4]:
print("Query 2: Fraud by Category")
spark.sql("""
    SELECT 
        category,
        COUNT(*) as total_transactions,
        SUM(is_fraud) as fraud_count,
        ROUND(AVG(amt), 2) as avg_amount,
        ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2) as fraud_rate
    FROM fraud_transactions
    GROUP BY category
    HAVING SUM(is_fraud) > 100
    ORDER BY fraud_count DESC
""").show()

Query 2: Fraud by Category


+--------------+------------------+-----------+----------+----------+
|      category|total_transactions|fraud_count|avg_amount|fraud_rate|
+--------------+------------------+-----------+----------+----------+
|   grocery_pos|            176191|       2228|    116.64|      1.26|
|  shopping_net|            139322|       2219|     86.94|      1.59|
|      misc_net|             90654|       1182|     80.18|      1.30|
|  shopping_pos|            166463|       1056|     78.91|      0.63|
| gas_transport|            188029|        772|     63.48|      0.41|
|      misc_pos|            114229|        322|     62.68|      0.28|
|     kids_pets|            161727|        304|     57.53|      0.19|
| entertainment|            134118|        292|     64.14|      0.22|
| personal_care|            130085|        290|     48.05|      0.22|
|          home|            175460|        265|     58.19|      0.15|
|   food_dining|            130729|        205|     50.99|      0.16|
|health_fitness|    

In [5]:
print("Query 3: Top 10 States by Fraud Count (RANK) ")
spark.sql("""
    SELECT 
        state,
        fraud_count,
        total_transactions,
        RANK() OVER (ORDER BY fraud_count DESC) as fraud_rank
    FROM (
        SELECT 
            state,
            SUM(is_fraud) as fraud_count,
            COUNT(*) as total_transactions
        FROM fraud_transactions
        GROUP BY state
    )
    ORDER BY fraud_rank
    LIMIT 10
""").show()

Query 3: Top 10 States by Fraud Count (RANK) 


+-----+-----------+------------------+----------+
|state|fraud_count|total_transactions|fraud_rank|
+-----+-----------+------------------+----------+
|   NY|        730|            119419|         1|
|   TX|        592|            135269|         2|
|   PA|        572|            114173|         3|
|   CA|        402|             80495|         4|
|   OH|        360|             66627|         5|
|   FL|        334|             60775|         6|
|   IL|        324|             62212|         7|
|   MI|        299|             65825|         8|
|   MN|        280|             45433|         9|
|   AL|        278|             58521|        10|
+-----+-----------+------------------+----------+



In [6]:
print(" Query 4: Fraud Rate by Amount Category (CASE WHEN) ")
spark.sql("""
    SELECT 
        amount_category,
        COUNT(*) as total,
        SUM(is_fraud) as fraud_count,
        ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2) as fraud_rate
    FROM (
        SELECT 
            is_fraud,
            CASE 
                WHEN amt < 100  THEN 'Low (< $100)'
                WHEN amt < 500  THEN 'Medium ($100-$500)'
                WHEN amt < 1000 THEN 'High ($500-$1000)'
                ELSE                 'Very High (> $1000)'
            END as amount_category
        FROM fraud_transactions
    )
    GROUP BY amount_category
    ORDER BY fraud_rate DESC
""").show()

 Query 4: Fraud Rate by Amount Category (CASE WHEN) 
+-------------------+-------+-----------+----------+
|    amount_category|  total|fraud_count|fraud_rate|
+-------------------+-------+-----------+----------+
|Very High (> $1000)|   5520|       1226|     22.21|
|  High ($500-$1000)|  16189|       3458|     21.36|
| Medium ($100-$500)| 313239|       2835|      0.91|
|       Low (< $100)|1517446|       2132|      0.14|
+-------------------+-------+-----------+----------+



In [7]:
print("Query 5: Top 10 Merchants with Most Fraud ")
spark.sql("""
    SELECT 
        merchant,
        fraud_count,
        total_transactions,
        ROUND(fraud_count * 100.0 / total_transactions, 2) as fraud_rate
    FROM (
        SELECT 
            merchant,
            SUM(is_fraud) as fraud_count,
            COUNT(*) as total_transactions
        FROM fraud_transactions
        GROUP BY merchant
    )
    WHERE fraud_count > 10
    ORDER BY fraud_count DESC
    LIMIT 10
""").show(truncate=False)

Query 5: Top 10 Merchants with Most Fraud 


+------------------------------+-----------+------------------+----------+
|merchant                      |fraud_count|total_transactions|fraud_rate|
+------------------------------+-----------+------------------+----------+
|fraud_Kilback LLC             |62         |6262              |0.99      |
|fraud_Rau and Sons            |60         |3546              |1.69      |
|fraud_Kozey-Boehm             |60         |2758              |2.18      |
|fraud_Doyle Ltd               |57         |3502              |1.63      |
|fraud_Terry-Huel              |56         |2864              |1.96      |
|fraud_Kuhn LLC                |55         |5031              |1.09      |
|fraud_Boyer PLC               |55         |4999              |1.10      |
|fraud_Kuhic LLC               |53         |2842              |1.86      |
|fraud_Kiehn-Emmerich          |53         |3574              |1.48      |
|fraud_Moen, Reinger and Murphy|53         |3393              |1.56      |
+------------------------

In [8]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

print("Query 6: UDF - Flag Suspicious Transactions ")

# Define UDF
def flag_transaction(amt, is_fraud):
    if is_fraud == 1 and amt > 500:
        return "HIGH RISK"
    elif is_fraud == 1:
        return "FRAUD"
    elif amt > 1000:
        return "SUSPICIOUS"
    else:
        return "NORMAL"

# Register UDF
flag_udf = udf(flag_transaction, StringType())

# Apply UDF to dataframe
df_flagged = df_full.withColumn(
    "risk_level",
    flag_udf("amt", "is_fraud")
)

# Register new view
df_flagged.createOrReplaceTempView("fraud_flagged")

# Query results
spark.sql("""
    SELECT 
        risk_level,
        COUNT(*) as count,
        ROUND(AVG(amt), 2) as avg_amount
    FROM fraud_flagged
    GROUP BY risk_level
    ORDER BY count DESC
""").show()

Query 6: UDF - Flag Suspicious Transactions 


+----------+-------+----------+
|risk_level|  count|avg_amount|
+----------+-------+----------+
|    NORMAL|1838449|     63.14|
|     FRAUD|   4967|    178.72|
| HIGH RISK|   4684|    903.86|
|SUSPICIOUS|   4294|   1999.98|
+----------+-------+----------+



In [9]:
print(" Query 7: Fraud Analysis by Gender ")
spark.sql("""
    SELECT 
        gender,
        COUNT(*) as total_transactions,
        SUM(is_fraud) as fraud_count,
        ROUND(AVG(amt), 2) as avg_transaction_amount,
        ROUND(MAX(amt), 2) as max_fraud_amount,
        ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2) as fraud_rate
    FROM fraud_transactions
    GROUP BY gender
    ORDER BY fraud_count DESC
""").show()

 Query 7: Fraud Analysis by Gender 
+------+------------------+-----------+----------------------+----------------+----------+
|gender|total_transactions|fraud_count|avg_transaction_amount|max_fraud_amount|fraud_rate|
+------+------------------+-----------+----------------------+----------------+----------+
|     F|           1014749|       4899|                 69.96|         28948.9|      0.48|
|     M|            837645|       4752|                 70.19|        27390.12|      0.57|
+------+------------------+-----------+----------------------+----------------+----------+



In [10]:
print("Query 8: Top 3 Highest Fraud Transactions per Category (ROW_NUMBER)")
spark.sql("""
    SELECT category, amt, merchant, trans_num, row_num
    FROM (
        SELECT 
            category,
            amt,
            merchant,
            trans_num,
            ROW_NUMBER() OVER (
                PARTITION BY category 
                ORDER BY amt DESC
            ) as row_num
        FROM fraud_transactions
        WHERE is_fraud = 1
    )
    WHERE row_num <= 3
    ORDER BY category, row_num
""").show(truncate=False)

Query 8: Top 3 Highest Fraud Transactions per Category (ROW_NUMBER)
+--------------+------+----------------------------------------+--------------------------------+-------+
|category      |amt   |merchant                                |trans_num                       |row_num|
+--------------+------+----------------------------------------+--------------------------------+-------+
|entertainment |695.53|fraud_Welch, Rath and Koepp             |420ea4195937516e902f33f14835ae02|1      |
|entertainment |687.06|fraud_Gibson-Deckow                     |a1d58c05d3c0cf1921e8e953f0ae4397|2      |
|entertainment |680.26|fraud_Kilback and Sons                  |c9ed6ee8c1b34a426b3c3e0cee11bddb|3      |
|food_dining   |149.53|fraud_Daugherty-Thompson                |76aad1df805831a6ef246d6ae1aab6e3|1      |
|food_dining   |149.42|fraud_O'Hara-Wilderman                  |f48450263599e23b91dc275b4462499f|2      |
|food_dining   |149.42|fraud_Schneider, Hayes and Nikolaus     |1cb66d986e7baf73cfae

In [11]:
print(" Metadata: Tables and Views ")
print("\nDatabases:")
for db in spark.catalog.listDatabases():
    print(f"  - {db.name}")

print("\nTables/Views:")
for table in spark.catalog.listTables():
    print(f"  - {table.name} (type: {table.tableType})")

print("\nColumns in fraud_transactions:")
for col in spark.catalog.listColumns("fraud_transactions"):
    print(f"  - {col.name}: {col.dataType}")

 Metadata: Tables and Views 

Databases:
  - default

Tables/Views:
  - fraud_flagged (type: TEMPORARY)
  - fraud_transactions (type: TEMPORARY)

Columns in fraud_transactions:
  - _c0: int
  - trans_date_trans_time: timestamp
  - cc_num: bigint
  - merchant: string
  - category: string
  - amt: double
  - first: string
  - last: string
  - gender: string
  - street: string
  - city: string
  - state: string
  - zip: int
  - lat: double
  - long: double
  - city_pop: int
  - job: string
  - dob: date
  - trans_num: string
  - unix_time: int
  - merch_lat: double
  - merch_long: double
  - is_fraud: int
